In [ ]:
import sqlite3
import os
import yaml
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing import event_accumulator
from collections import defaultdict

def get_nebula_to_title_mapping(db_path):
    """
    Connects to the scenarios.db and retrieves a mapping of nebula_id to title.

    Args:
        db_path (str): The full path to the scenarios.db SQLite database file.

    Returns:
        dict: A dictionary where keys are the nebula_ids (names) and values
              are the corresponding titles. Returns an empty dictionary on error.
    """
    if not os.path.exists(db_path):
        print(f"Error: Database file not found at '{db_path}'")
        return {}

    mapping = {}
    conn = None
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT name, title FROM scenarios")
        rows = cursor.fetchall()
        for row in rows:
            nebula_id, title = row
            mapping[nebula_id] = title
    except sqlite3.Error as e:
        print(f"Database error: {e}")
    finally:
        if conn:
            conn.close()
    return mapping

def plot_experiment_comparison(experiment_dirs, title_mapping):
    """
    Compares multiple experiments by plotting an aggregated metric from their participants.
    Uses a title mapping to create human-readable labels for the plot legend.

    Args:
        experiment_dirs (list[str]): List of paths to experiment directories.
        title_mapping (dict): A dictionary mapping experiment names (nebula_ids) to titles.
    """
    all_experiments_metrics = {}  # {tag: {exp_name: (steps, agg_values)}}

    print("--- Processing and comparing multiple experiments ---")

    # --- 1. Loop through each experiment directory to gather data ---
    for exp_dir in experiment_dirs:
        exp_name = os.path.basename(exp_dir)
        metrics_dir = os.path.join(exp_dir, 'metrics')

        if not os.path.isdir(metrics_dir):
            print(f"  - Skipping '{exp_name}': 'metrics' directory not found.")
            continue

        print(f"Processing experiment: {exp_name}")
        participant_dirs = [
            os.path.join(metrics_dir, d) for d in os.listdir(metrics_dir)
            if os.path.isdir(os.path.join(metrics_dir, d)) and d.startswith('participant_')
        ]
        if not participant_dirs:
            print(f"  - No participant directories found in '{metrics_dir}'.")
            continue

        # --- 2. Collect and aggregate data from all participants in the experiment ---
        experiment_data_by_tag = defaultdict(list)
        for p_dir in participant_dirs:
            try:
                ea = event_accumulator.EventAccumulator(p_dir, size_guidance={event_accumulator.SCALARS: 0})
                ea.Reload()
                for tag in ea.Tags()['scalars']:
                    events = ea.Scalars(tag)
                    steps = [e.step for e in events]
                    values = [e.value for e in events]
                    experiment_data_by_tag[tag].append((steps, values))
            except Exception as e:
                p_name = os.path.basename(p_dir)
                print(f"    - Error loading data for {p_name}: {e}")

        for tag, participant_runs in experiment_data_by_tag.items():
            step_to_values = defaultdict(list)
            for steps, values in participant_runs:
                for step, value in zip(steps, values):
                    step_to_values[step].append(value)
            if not step_to_values: continue
            sorted_steps = sorted(step_to_values.keys())
            agg_func = min if 'loss' in tag.lower() else max
            aggregated_values = [agg_func(step_to_values[step]) for step in sorted_steps]
            if tag not in all_experiments_metrics:
                all_experiments_metrics[tag] = {}
            all_experiments_metrics[tag][exp_name] = (sorted_steps, aggregated_values)

    if not all_experiments_metrics:
        print("\nNo metrics were found across any of the experiments.")
        return

    # --- 3. Plot the aggregated data using titles from the mapping ---
    print("\n--- Generating experiment comparison plots ---")
    for tag, experiments_data in all_experiments_metrics.items():
        plt.figure(figsize=(12, 8))
        agg_type = "MIN" if 'loss' in tag.lower() else "MAX"
        plt.title(f'Experiment Comparison: Best Participant Performance ({agg_type}) for "{tag}"')
        plt.xlabel('Step')
        plt.ylabel('Aggregated Value')

        for exp_name, (steps, values) in experiments_data.items():
            # Use the title from the mapping for the legend, or the ID as a fallback
            display_name = title_mapping.get(exp_name, exp_name)
            plt.plot(steps, values, label=display_name, alpha=0.9)

        plt.grid(True, which='both', linestyle='--', linewidth=0.5)
        plt.legend(title="Experiment Title")
        plt.tight_layout()

    print("\nDisplaying plots...")
    plt.show()
    print("--- Finished processing. ---")

In [ ]:
database_path = '../app/databases/scenarios.db'
base_path = '../app/logs/'

In [ ]:
import glob
import os

search_pattern = os.path.join(base_path, 'nebula_DFL_*')

print(f"Searching for directories with pattern: {search_pattern}")

# Use glob.glob() to find all files and directories matching the pattern.
all_matching_paths = glob.glob(search_pattern)

# Filter the list to include only directories, not files.
# os.path.isdir() checks if a given path is a directory.
experiment_paths = [path for path in all_matching_paths if os.path.isdir(path)]

# Sort the paths for consistency (optional, but good practice).
experiment_paths.sort()

# Print the list of found experiment directories.
print("\nFound experiment directories:")
print(f"Total: {len(experiment_paths)}")
# if experiment_paths:
#     for path in experiment_paths:
#         print(path)
# else:
#     print("No matching directories found.")


In [ ]:



if __name__ == '__main__':
    # --- USAGE EXAMPLE ---
    # STEP 1: Set the path to your scenarios database file.

    # --- Script execution starts here ---
    print("--- Step 1: Reading experiment titles from database ---")
    nebula_title_map = get_nebula_to_title_mapping(database_path)

    if not nebula_title_map:
        print("\nCould not retrieve mapping. Please check the database path.")
        print("Continuing with experiment IDs instead of titles.")

    valid_experiment_paths = [p for p in experiment_paths if os.path.isdir(p)]

    if not experiment_paths or not valid_experiment_paths:
        print("\n" + "="*50)
        print("WARNING: No valid experiment directories were provided.")
        print("Please update the 'experiment_paths' list in the script.")
        print("="*50)
    else:
        print("\n--- Step 2: Plotting experiment data ---")
        plot_experiment_comparison(valid_experiment_paths, nebula_title_map)
